# Lean 4 verification: spectral stability of strong ideal-gas shock layers

This notebook rebuilds and checks the Lean 4 formalization that accompanies the manuscript *Spectral stability for strong ideal-gas shock layers*. It depends only on Mathlib v4.34.1 and Lean v4.34.1. Run the cells from top to bottom. The sources are embedded in the notebook, so nothing needs to be uploaded or placed by hand, and a re-run skips the steps that have already finished. Section numbers below refer to the manuscript.

**Checked in Lean from first principles**

* The cancellation of the thermal mass coupling by the change of variables `h = ε + c u` (Section 4).
* The weighted pointwise energy identity of Proposition 5.1 and Appendix A, for an arbitrary thermal weight, including the formula (5.10) for the coupling coefficient.
* The explicit limiting profile of Lemma 2.1, which also shows that the profile hypotheses can be satisfied.
* Lemma 2.2: the slope interval, the sharp bounds above the reference line and below `ν = 1`, the downstream slope relation (6.7), and the upstream limit of the slope.
* Section 6: Lemma 6.1 and Proposition 6.2 for the constant weight, the interpolation identity of Proposition 6.3, the coercivity of Proposition 6.5 for the variable weight, Lemma 6.4 on the endpoint polynomial, and the equivalence of `Γ ≥ Γ_E(ν)` with the endpoint condition (6.16).
* Lemma 7.1, including the upstream degeneration of the variable weight for `2/3 < ν < 1`, and the exclusion of nonzero frequencies (Proposition 7.2).
* The zero-frequency argument (Proposition 8.1).
* Theorem 1.1 on the region `ℛ`, and the compactness argument of Section 9 that gives Corollary 1.2 for every compact subset of `ℛ`.
* The membership of air and of monatomic gases in `ℛ` (Appendix C).

**Entering as explicit hypotheses.** Results of Humpherys, Lyng, and Zumbrun (Arch. Ration. Mech. Anal., 2009) that are outside the present scope of Mathlib. These are the existence of the limiting profile with its endstate limits and a positive downstream slope limit, the exponential decay of admissible solutions, the continued outgoing space at zero frequency, the saddle structure of the downstream rest point, and, for finite shocks, the continuity, high-frequency bound, and characterization of zeros of the integrated Evans function. They appear as hypotheses of the theorems, never as axioms.

The audit cell confirms that the sources contain no `sorry` and that every main result relies only on Lean's three standard axioms.


In [ ]:
# 1. Install elan (the Lean toolchain manager). About 30 s; skipped if already installed.
import subprocess, sys, time, os
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]
def run(cmd, cwd="/content/StrongShock", check=True, show_all=False):
    """Run a shell command, streaming its output."""
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if show_all or not line.startswith(("warning", "Note:", "Hint:", "  [apply]", "  ")):
            print(line, end="")
    p.wait()
    print(f"[exit {p.returncode}, {time.time()-t0:.0f} s]")
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return "".join(lines)

if subprocess.run("command -v elan", shell=True).returncode != 0:
    run("curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -o /tmp/elan-init.sh"
        " && bash /tmp/elan-init.sh -y --default-toolchain none", cwd="/tmp")
else:
    print("elan already installed")
_ = run("elan --version", cwd="/tmp")


In [ ]:
# 2. Unpack the formalization (embedded as base64; nothing to upload).
#    Re-running rewrites the sources but keeps the downloaded Mathlib and the build cache.
import base64, io, tarfile, os
PAYLOAD = "..."
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz") as tf:
    try:
        tf.extractall("/content", filter="data")
    except TypeError:
        tf.extractall("/content")
n = sum(len(fn) for _, _, fn in os.walk("/content/StrongShock") if ".lake" not in _)
print(n, "source files in /content/StrongShock")
print(open("/content/StrongShock/README.md").read())


In [ ]:
# 3. Fetch Mathlib v4.34.1 (pinned in lake-manifest.json) and its prebuilt binary cache.
#    Skipped when Mathlib is already present from an earlier run.
mathlib = "/content/StrongShock/.lake/packages/mathlib"
marker = "/content/StrongShock/.lake/cache_done"
if not os.path.isdir(mathlib):
    _ = run("lake update")
if not os.path.exists(marker):
    _ = run("lake exe cache get")
    open(marker, "w").write("ok")
else:
    print("Mathlib cache already in place")


In [ ]:
# 4. Build and check every proof. About 2-5 minutes depending on the runtime's cores;
#    a re-run only rebuilds files that changed.
log = run("lake build", check=False)
assert "error:" not in log and "Build completed successfully" in log, "build failed"
print("BUILD OK")


In [ ]:
# 5. Audit: no sorry/admit/axiom in the sources, and the axioms behind the main theorems.
import re, glob
srcs = glob.glob("/content/StrongShock/StrongShock/**/*.lean", recursive=True) + ["/content/StrongShock/StrongShock.lean"]
bad = [f for f in srcs if re.search(r"\bsorry\b|\badmit\b|^\s*axiom\s", open(f).read(), re.M)]
print("source files with sorry/admit/axiom:", bad if bad else "none")
out = run("lake env lean Check.lean", check=True, show_all=True)
std = "[propext, Classical.choice, Quot.sound]"
lines = [l for l in out.splitlines() if "depends on axioms" in l]
ok = (not bad) and len(lines) > 0 and all(l.endswith(std) for l in lines)
print("\nRESULT:", f"{len(lines)} main results checked; all use only {std}" if ok else "CHECK FAILED")


In [ ]:
# 6. Print the statements of the main results as Lean sees them.
stmts = """import StrongShock
open StrongShock
#check @InR
#print GammaE
#print rhoE
#print PiP
#check @thm_nonzero
#check @evans_R
#check @cor_finite
#check @monatomic_in_R
"""
open("/content/StrongShock/Statements.lean", "w").write(stmts)
_ = run("lake env lean Statements.lean", check=True, show_all=True)
